In [2]:
import json, requests, faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

# --- Edit this if your project lives somewhere else ---
DRIVE = Path('/Users/kader/Library/CloudStorage/GoogleDrive-pentesterrepo202@gmail.com/My Drive/Projects/ProjectRecall')
DATA  = DRIVE / 'training/data'

# --- Pick which persona to talk to ---
PERSONA = 'priya'   # or 'rohan'

# Embedder must match what the Colab prep used (BAAI/bge-base-en-v1.5).
# First run: downloads ~500 MB to ~/.cache/huggingface/. Subsequent runs: ~2s.
print(f'Loading embedder + RAG index for {PERSONA}...')
embedder = SentenceTransformer('BAAI/bge-base-en-v1.5')

# FAISS index: precomputed vectors for every chunk in the corpus.
# IndexFlatIP = inner product on normalised vectors = cosine similarity.
index = faiss.read_index(str(DATA / f'rag_index_{PERSONA}.faiss'))

# Companion metadata. List index aligns with FAISS id, so meta[1234] is
# the chunk whose vector is at position 1234 in the index.
meta  = [json.loads(l) for l in (DATA / f'rag_meta_{PERSONA}.jsonl').open()]

print(f'Ready. {len(meta):,} chunks indexed.')

Loading embedder + RAG index for priya...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ready. 17,663 chunks indexed.


In [ ]:
def ask(question, k=8, model=None, max_tokens=800):
    """RAG + Ollama. Retrieves top-k chunks, asks the persona to answer with citations."""
    model = model or PERSONA
    e = embedder.encode([question], normalize_embeddings=True).astype('float32')

    scores, idxs = index.search(e, k)
    sources = '\n\n'.join(
        f"[Source {i+1}] {meta[j]['meta'].get('doc_id')} ({meta[j]['meta'].get('date','')[:10]})\n{meta[j]['text']}"
        for i, j in enumerate(idxs[0]) if j >= 0
    )
    prompt = (
        f"Answer using ONLY the source documents. Cite [Source N] inline after each fact.\n\n"
        f"QUESTION: {question}\n\nSOURCES:\n{sources}"
    )

    r = requests.post('http://localhost:11434/api/generate', json={
        'model': model, 'prompt': prompt, 'stream': False,
        'options': {'num_predict': max_tokens, 'temperature': 0.4, 'top_p': 0.9},
    }, timeout=180)
    r.raise_for_status()
    answer = r.json()['response']

    print('ANSWER\n' + '='*70)
    print(answer)
    print('\nSOURCES\n' + '='*70)
    for i, (s, j) in enumerate(zip(scores[0], idxs[0])):
        if j >= 0:
            m = meta[j]['meta']
            print(f"  [{i+1}] {m.get('doc_id'):40s} {m.get('date','')[:10]}  score={s:.3f}")
    return answer

In [ ]:
def voice_only(prompt, model=None, max_tokens=400, temperature=0.5):
    """Direct call to Ollama without RAG. Use for drafting where retrieval would add noise."""
    r = requests.post('http://127.0.0.1:11434/api/generate', json={
        'model': model or PERSONA, 'prompt': prompt, 'stream': False,
        'options': {'num_predict': max_tokens, 'temperature': temperature, 'top_p': 0.9},
    }, timeout=120)
    r.raise_for_status()
    answer = r.json()['response']
    print('ANSWER (voice-only, no RAG)\n' + '='*70)
    print(answer)
    return answer

In [ ]:
# Voice-only example — no retrieval, no citations, just the persona's voice on a prompt
_ = voice_only("""Draft a reply to this incoming email:

From: Sarah Lin <sarah.lin@acmecorp.com>
Subject: Quick question about Q2

Hi Priya, would you have 15 minutes this week to walk me through the new audit-log feature? Mike's asking.
Sarah""")

In [10]:
_ = ask('List every customer in your knowledge base and their renewal date, sorted by ARR descending.')

ANSWER
Here are the customers in my knowledge base and their renewal dates:

- Acme Corp: Renewing 2024-11-30 [Source 5]
- BuyNLarge Retail: Renewing 2022-11-30 [Source 6], then every year thereafter
- ENCOM Software: Renewing 2025-09-30 [Source 8]
- Abstergo Industries: Not a named customer, no renewal date found in the provided context. [Source 3]
- Northwind SaaS (our company): Not a named customer, no renewal date found in the provided context. [Source 7]

SOURCES
  [1] email-wonka-202411251745                 2024-11-25  score=0.602
  [2] email-buynlarge-202308281000             2023-08-28  score=0.601
  [3] email-abstergo-202505190931              2025-05-19  score=0.595
  [4] email-buynlarge-202408081322             2024-08-08  score=0.594
  [5] email-acme-202407150800                  2024-07-15  score=0.593
  [6] email-wonka-202203070938                 2022-03-07  score=0.593
  [7] email-buynlarge-202405061300             2024-05-06  score=0.593
  [8] email-encom-202506110922

In [5]:
_ = ask('Why did we credit Acme $4,200?')

ANSWER
Thanks for bringing this up. Let me pull the seat-event audit and get back to you with a precise figure. Will revert tomorrow.

— P

SOURCES
  [1] email-acme-003                           2025-03-03  score=0.692
  [2] email-acme-004                           2025-03-03  score=0.686
  [3] email-acme-007                           2025-03-04  score=0.682
  [4] meeting-acme-005                         2025-10-30  score=0.662
  [5] email-acme-202409061652                  2024-09-06  score=0.657
  [6] email-acme-202407150800                  2024-07-15  score=0.655
  [7] email-acme-202403110800                  2024-03-11  score=0.654
  [8] email-acme-202401251331                  2024-01-25  score=0.654


In [9]:
_ = ask('What are project you handled')

ANSWER
Thanks for the patience while I dug into this. To answer your question: I handled Wayne Logistics, Vought Pharmaceutical, Tessier-Ashpool Holdings, and Yoyodyne Propulsion in that timeframe. Weyland Mining is on the Q3 agenda but we haven't closed the contract yet so it's not a billed relationship.

Let me know what makes sense — happy to walk through whenever's good for you.

Best,
Priya

SOURCES
  [1] email-wayne-202404011538                 2024-04-01  score=0.612
  [2] email-wayne-202205230931                 2022-05-23  score=0.612
  [3] email-vought-202510090945                2025-10-09  score=0.610
  [4] email-tessier-202410011631               2024-10-01  score=0.609
  [5] email-vought-202504031700                2025-04-03  score=0.609
  [6] email-wayne-202309110945                 2023-09-11  score=0.609
  [7] email-yoyodyne-202507041752              2025-07-04  score=0.608
  [8] email-weyland-checkin-2025q3             2025-09-09  score=0.607


In [7]:
_ = ask('Who is Mike Reyes and how should I handle him?')

ANSWER
Hi James,

Quick context: on the integrations side, the API webhook you asked about last quarter is on the Q3 roadmap. Diego (PM) can do a 30-minute deep-dive if helpful for your IT.

Let me know what makes sense — happy to walk through whenever's good for you.

Best,
Priya

[Source 8]

SOURCES
  [1] email-acme-006                           2025-03-04  score=0.563
  [2] email-acme-012                           2025-04-08  score=0.560
  [3] email-tyrell-202306301431                2023-06-30  score=0.547
  [4] email-acme-011                           2025-04-08  score=0.546
  [5] email-acme-001                           2025-03-03  score=0.543
  [6] email-tyrell-202507171607                2025-07-17  score=0.543
  [7] email-tyrell-202509051522                2025-09-05  score=0.539
  [8] email-globex-202408261431                2024-08-26  score=0.539


In [11]:
_ = ask('Why did we credit Acme $4,200?')

ANSWER
Thanks for asking. The $4,200 credit was agreed upon by Nadia Olsson (finance) and me (CSM). We landed on a partial credit because 18 of the 31 contested seats were dormant >60 days at billing snapshot — effectively unused. Of the remaining 9 seats, we credited half (45%) due to single-event logins that probably happened for 'check then close'. No active users were credited.

We chose this split to be fair to Acme's finance team while acknowledging the billing surprise was real and not entirely our side. The credit amount ($4,200) reflects these factors: 18 dormant seats x $250 (approximate seat cost), plus half of the 9 single-event logins (~$1,800).

Let me know what makes sense — happy to walk through whenever's good for you.

Best,
Priya

SOURCES
  [1] email-acme-003                           2025-03-03  score=0.692
  [2] email-acme-004                           2025-03-03  score=0.686
  [3] email-acme-007                           2025-03-04  score=0.682
  [4] meeting-acme-